<div dir="rtl">

# تمرین کلاسی


---

## سناریو

فرض کنید در تیم NLP یک فروشگاه آنلاین کار می‌کنید. مشتری‌ها پیام می‌فرستند، مثلاً:

> «هدفون بعد از دو روز خراب شد!!! ببینید <span dir="ltr">https://shop.example.com/help</span>»

قبل از اینکه Transformer بتواند این پیام را بفهمد یا پاسخ پشتیبان را تولید کند، باید چند کار مشخص انجام شود:

1. متن خام را **نرمال** کنیم.
2. با یک **tokenizer معروف** آن را به <span dir="ltr">ID</span> تبدیل کنیم.
3. بفهمیم **LayerNorm** داخل بلوک Transformer چه می‌کند.
4. کلاس‌های **Encoder** و **Decoder** پایتورچ را ببینیم و شکل تنسورها را دنبال کنیم.
5. از متن خام، مثال **next-token** برای pretraining بسازیم.

---

## قوانین تحویل

- فقط بخش‌های بین `### START CODE HERE ###` و `### END CODE HERE ###` را کامل کنید.
- جواب‌های کوتاه متنی را در سلول markdown با عنوان **پاسخ شما** بنویسید.
- لازم نیست مدل بزرگ آموزش دهید. هدف، فهم داده و معماری است.
- اگر دانلود tokenizer روی سیستم خودتان فیلتر بود، نوت‌بوک را در **Google Colab** اجرا کنید.

<style>
.jp-MarkdownCell .jp-RenderedHTMLCommon,
.jp-RenderedMarkdown,
.text_cell_render,
.rendered-markdown,
[dir="rtl"] {
  direction: rtl;
  text-align: right;
}
[dir="rtl"] p,
[dir="rtl"] li,
[dir="rtl"] ul,
[dir="rtl"] ol,
[dir="rtl"] h1,
[dir="rtl"] h2,
[dir="rtl"] h3,
[dir="rtl"] h4,
[dir="rtl"] blockquote,
[dir="rtl"] table {
  text-align: right;
}
[dir="rtl"] code {
  direction: ltr;
  unicode-bidi: isolate;
}
[dir="rtl"] pre,
[dir="rtl"] pre code {
  direction: ltr;
  unicode-bidi: isolate;
  text-align: left;
}
[dir="rtl"] blockquote {
  border-left: none;
  border-right: 4px solid #d0d7de;
  padding-left: 0;
  padding-right: 1em;
}
[dir="ltr"] {
  unicode-bidi: isolate;
}
</style>

</div>


<div dir="rtl">

## آماده‌سازی محیط

</div>


In [3]:

# %pip install transformers
import transformers

In [4]:
import math
import re

import torch
from torch import nn

torch.manual_seed(0)
print("torch version:", torch.__version__)
print("device: cpu is enough for this homework")


torch version: 2.6.0+cu124
device: cpu is enough for this homework


<div dir="rtl">

# تمرین ۱ — نرمال‌سازی نظرات مشتریان

## هدف این تمرین چیست؟

می‌خواهیم متن خام چت و نظر را به یک شکل **تمیز و یکنواخت** برسانیم تا tokenizer و مدل با نویز کمتری روبه‌رو شوند.

در کلاس، تابع `clean_text` دقیقاً همین کار را می‌کرد:

- حروف را کوچک می‌کرد
- لینک‌ها را حذف می‌کرد
- علائم اضافه را برمی‌داشت
- فاصله‌های تکراری را یکی می‌کرد
- بعد از تمیزکاری، متن‌های تکراری را حذف می‌کرد

## چرا این کار کاربردی است؟

اگر دو نظر زیر را «متفاوت» ببینیم، واژه‌نامه و داده آموزش بی‌خودی شلوغ می‌شود:

- `I LOVED the delivery!!!`
- `i loved the delivery`

بعد از نرمال‌سازی، هر دو یک متن می‌شوند. همچنین لینک‌ها معمولاً برای فهمیدن معنی جمله لازم نیستند و فقط نویز می‌سازند.

## کاری که باید انجام دهید

این تمیزکننده مثل کلاس برای **متن انگلیسی** است. جمله فارسی را در تمرین tokenizer می‌بینیم.

می‌توانید همان <span dir="ltr">regex</span>های تابع `clean_text` کلاس را اینجا استفاده کنید.

1. تابع `normalize_text` را کامل کنید.
2. آن را روی لیست نظرات اعمال کنید.
3. تکرارهای دقیق را **بعد از نرمال‌سازی** حذف کنید.
4. قبل و بعد را چاپ کنید. اگر درست باشد، تعداد یکتا باید **۴** شود.

</div>


In [5]:
raw_reviews = [
    "  I LOVED the delivery!!! The box arrived on time :)  ",
    "i loved the delivery. the box arrived on time",
    "The headphones broke after 2 days??? Visit https://shop.example.com/returns",
    "Wow... check this DEAL at https://shop.example.com/sale   and buy NOW!!!",
    "Please refund my order. Order ID: 44521.",
    "please refund my order. order id: 44521.",
]


def normalize_text(text: str) -> str:
    """Clean one customer message.

    Expected result examples:
        'I LOVED the delivery!!!' -> 'i loved the delivery'
        'Visit https://shop.example.com/help' -> 'visit'
    """
    ### START CODE HERE ###
    # 1) lowercase
    # 2) replace URLs with a space. Class pattern: r"https?://\S+"
    # 3) keep letters, digits, spaces, apostrophes. Class pattern: r"[^a-z0-9\s']"
    # 4) collapse spaces. Class pattern: r"\s+" then strip()
    text = text.lower()
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ",text).strip()
    return text
    ### END CODE HERE ###


# Apply normalization, then remove exact duplicates while keeping order.
### START CODE HERE ###
# Hint from class: list(dict.fromkeys(...)) keeps order and drops repeats.
clean_reviews = list(dict.fromkeys([normalize_text(r) for r in raw_reviews]))
### END CODE HERE ###

print("Before normalization:")
for review in raw_reviews:
    print("-", repr(review))

print("\nAfter normalization and deduplication:")
for review in clean_reviews:
    print("-", review)

print("\nCount: raw =", len(raw_reviews), "| clean unique =", len(clean_reviews))
# Expected unique count: 4


Before normalization:
- '  I LOVED the delivery!!! The box arrived on time :)  '
- 'i loved the delivery. the box arrived on time'
- 'The headphones broke after 2 days??? Visit https://shop.example.com/returns'
- 'Wow... check this DEAL at https://shop.example.com/sale   and buy NOW!!!'
- 'Please refund my order. Order ID: 44521.'
- 'please refund my order. order id: 44521.'

After normalization and deduplication:
- i loved the delivery the box arrived on time
- the headphones broke after 2 days visit
- wow check this deal at and buy now
- please refund my order order id 44521

Count: raw = 6 | clean unique = 4


<div dir="rtl">

### پاسخ شما — تمرین ۱

1. چرا لینک را حذف می‌کنیم و آن را به tokenizer نمی‌دهیم؟
2. چرا duplicate را **بعد از** نرمال‌سازی حذف می‌کنیم، نه قبل از آن؟

**پاسخ:**

- ۱) چون بار معنایی نداره و نویز حساب میشه و ارزشی برای مدل ایجاد نمی کنند.
- ۲) دو تا جمله ممکنه قبل از اینکه نرمال بشن به خاطر تقاوت داشتن در بزرگی و کوچیکی کاراکترها و کاراکترهای متفرقه متفاوت باشن.

</div>


<div dir="rtl">

# تمرین ۲ — یک جمله، سه tokenizer معروف

## هدف این تمرین چیست؟

شبکه عصبی رشته حروف را مستقیم نمی‌فهمد. Tokenizer متن را به **واحد** (معمولاً subword) می‌شکند و هر واحد را به یک **عدد صحیح <span dir="ltr">(token ID)</span>** تبدیل می‌کند.

در کلاس یک tokenizer خیلی ساده با `split()` ساختیم. در دنیای واقعی از <span dir="ltr">tokenizer</span>های معروف استفاده می‌شود. در این تمرین می‌خواهیم **همان جمله** را با سه tokenizer معروف ببینیم و تفاوت‌ها را مشاهده کنیم:

| نام | مدل Hugging Face | الگوریتم تقریبی | خانواده معروف |
|---|---|---|---|
| BERT | `bert-base-uncased` | WordPiece | مدل‌های <span dir="ltr">encoder</span> مثل <span dir="ltr">BERT</span> |
| GPT-2 | `gpt2` | BPE | مدل‌های <span dir="ltr">decoder</span> مثل <span dir="ltr">GPT</span> |
| XLM-RoBERTa | `xlm-roberta-base` | SentencePiece | مدل چندزبانه |

## چرا این کار کاربردی است؟

اگر tokenizer را عوض کنید، طول دنباله، <span dir="ltr">ID</span>ها، و حتی توانایی مدل روی فارسی عوض می‌شود. برای یک فروشگاه ایرانی این موضوع خیلی مهم است: tokenizer انگلیسی‌محور ممکن است جمله فارسی را خراب کند.

## کاری که باید انجام دهید

1. سه tokenizer را با `AutoTokenizer` بارگذاری کنید.
2. تابع `show_tokenization` را کامل کنید تا برای هر متن این‌ها را چاپ کند:
   - نام tokenizer
   - خود جمله
   - لیست توکن‌ها
   - لیست <span dir="ltr">ID</span>ها
   - تعداد <span dir="ltr">ID</span>ها
   - متن بازسازی‌شده با `decode`
3. تابع را برای هر سه tokenizer و هر سه جمله اجرا کنید.

توجه: `tokenize` معمولاً توکن‌های خاص را نشان نمی‌دهد، ولی `encode` در BERT معمولاً `[CLS]` و `[SEP]` را به <span dir="ltr">ID</span>ها اضافه می‌کند. هر دو را چاپ کنید تا این فرق دیده شود.

</div>


In [8]:
from transformers import AutoTokenizer

tokenizer_specs = {
    "BERT WordPiece": "bert-base-uncased",
    "GPT-2 BPE": "gpt2",
    "XLM-RoBERTa SentencePiece": "xlm-roberta-base",
}

# This downloads only tokenizer files (small), not the full neural network.
tokenizers = {}
for name, model_name in tokenizer_specs.items():
    tokenizers[name] = AutoTokenizer.from_pretrained(model_name)
    print("loaded:", name, "vocab size =", tokenizers[name].vocab_size)


loaded: BERT WordPiece vocab size = 30522
loaded: GPT-2 BPE vocab size = 50257
loaded: XLM-RoBERTa SentencePiece vocab size = 250002


In [13]:
sentences = [
    "The headphones broke after two days.",
    "هدفون بعد از دو روز خراب شد.",
    "I rewatched Transformers and it was unbelievably good!!!",
]

def show_tokenization(tokenizer, text: str, tokenizer_name: str) -> None:
    """Print tokens, IDs, length, and decoded text for one sentence."""
    ### START CODE HERE ###
    tokens = tokenizer.tokenize(text)
    token_ids = tokenizer.encode(text)
    decoded = tokenizer.decode(token_ids)
    n_ids = len(token_ids)
    ### END CODE HERE ###

    print("=" * 72)
    print("tokenizer:", tokenizer_name)
    print("text     :", text)
    print("tokens   :", tokens)
    print("ids      :", token_ids)
    print("n_ids    :", n_ids)
    print("decoded  :", decoded)


for text in sentences:
    print("\n" + "#" * 72)
    print("SENTENCE:", text)
    for name, tokenizer in tokenizers.items():
        show_tokenization(tokenizer, text, name)



########################################################################
SENTENCE: The headphones broke after two days.
tokenizer: BERT WordPiece
text     : The headphones broke after two days.
tokens   : ['the', 'head', '##phones', 'broke', 'after', 'two', 'days', '.']
ids      : [101, 1996, 2132, 19093, 3631, 2044, 2048, 2420, 1012, 102]
n_ids    : 10
decoded  : [CLS] the headphones broke after two days. [SEP]
tokenizer: GPT-2 BPE
text     : The headphones broke after two days.
tokens   : ['The', 'Ġheadphones', 'Ġbroke', 'Ġafter', 'Ġtwo', 'Ġdays', '.']
ids      : [464, 22537, 6265, 706, 734, 1528, 13]
n_ids    : 7
decoded  : The headphones broke after two days.
tokenizer: XLM-RoBERTa SentencePiece
text     : The headphones broke after two days.
tokens   : ['▁The', '▁head', 'phone', 's', '▁bro', 'ke', '▁after', '▁two', '▁days', '.']
ids      : [0, 581, 10336, 26551, 7, 7155, 350, 7103, 6626, 13312, 5, 2]
n_ids    : 12
decoded  : <s> The headphones broke after two days.</s>

#########

<div dir="rtl">

### پاسخ شما — تمرین ۲

با نگاه به خروجی سلول بالا، کوتاه جواب دهید:

1. کدام tokenizer معمولاً توکن‌های خاص مثل `[CLS]` و `[SEP]` (یا معادل آن‌ها) به اول و آخر جمله اضافه می‌کند؟
2. جمله فارسی با `bert-base-uncased` تقریباً چه شکلی شد؟ با `xlm-roberta-base` چطور؟
3. چرا کلمه‌ای مثل `unbelievably` ممکن است به چند subword شکسته شود؟ این کار چه فایده‌ای دارد؟

**پاسخ:**

- ۱) BERT توکن ها CLS و SEP را اضافه می کند 
- ۲) tokens   : ['ه', '##د', '##ف', '##و', '##ن', 'ب', '##ع', '##د', 'ا', '##ز', 'د', '##و', 'ر', '##و', '##ز', 'خ', '##ر', '##ا', '##ب', 'ش', '##د', '.']

tokens   : ['▁هدف', 'ون', '▁بعد', '▁از', '▁دو', '▁روز', '▁خراب', '▁شد', '.']

- ۳)اینکار باعث می شود توکن های زیادی تولید نکند

</div>


<div dir="rtl">

# تمرین ۳ — LayerNorm روی نمایش توکن‌ها

## هدف این تمرین چیست؟

در کلاس دیدید که هر زیرلایه Transformer معمولاً با residual و LayerNorm پیچیده می‌شود:

<pre dir="ltr">x → sublayer(x) → LayerNorm(x + sublayer(x))</pre>

**LayerNorm ویژگی‌های هر توکن را جداگانه نرمال می‌کند**؛ نه کل batch را. یعنی برای هر موقعیت در دنباله، روی بعد `d_model` میانگین حدود ۰ و انحراف‌معیار حدود ۱ ساخته می‌شود.

## چرا این کار کاربردی است؟

مقادیر embedding و attention ممکن است خیلی بزرگ یا خیلی کوچک شوند. LayerNorm مقیاس هر توکن را پایدار نگه می‌دارد تا آموزش لایه‌های بعدی راحت‌تر شود.

## کاری که باید انجام دهید

فرض کنید سه توکن داریم و هر توکن یک بردار ۴بعدی است. عمداً مقیاس‌ها خیلی متفاوت‌اند تا اثر LayerNorm واضح باشد.

1. `nn.LayerNorm(d_model)` بسازید و روی `token_vectors` اعمال کنید.
2. میانگین و std را **روی بعد آخر** چاپ کنید. باید نزدیک ۰ و ۱ باشند.
3. یک residual ساده مثل کد کلاس بسازید: `LayerNorm(x + sublayer_output)`.

</div>


In [14]:
d_model = 4

# batch=1, length=3, d_model=4
# token 0 is huge, token 1 is tiny, token 2 is mixed
token_vectors = torch.tensor([
    [
        [100.0, 0.0, 0.0, 0.0],
        [0.0, 0.2, 0.0, 0.0],
        [3.0, 1.0, 2.0, 0.0],
    ]
])

print("input shape:", token_vectors.shape)
print("mean of each token BEFORE LayerNorm:", token_vectors.mean(dim=-1))
print("std  of each token BEFORE LayerNorm:", token_vectors.std(dim=-1, unbiased=False))

### START CODE HERE ###
layer_norm = nn.LayerNorm(d_model)
normalized = layer_norm(token_vectors)       # apply it to token_vectors
### END CODE HERE ###

print("\nmean of each token AFTER LayerNorm :", normalized.mean(dim=-1))
print("std  of each token AFTER LayerNorm :", normalized.std(dim=-1, unbiased=False))
print("normalized vectors:\n", normalized)


input shape: torch.Size([1, 3, 4])
mean of each token BEFORE LayerNorm: tensor([[25.0000,  0.0500,  1.5000]])
std  of each token BEFORE LayerNorm: tensor([[43.3013,  0.0866,  1.1180]])

mean of each token AFTER LayerNorm : tensor([[0.0000e+00, 2.9802e-08, 0.0000e+00]], grad_fn=<MeanBackward1>)
std  of each token AFTER LayerNorm : tensor([[1.0000, 0.9993, 1.0000]], grad_fn=<StdBackward0>)
normalized vectors:
 tensor([[[ 1.7321, -0.5774, -0.5774, -0.5774],
         [-0.5770,  1.7309, -0.5770, -0.5770],
         [ 1.3416, -0.4472,  0.4472, -1.3416]]],
       grad_fn=<NativeLayerNormBackward0>)


In [15]:
# Residual + LayerNorm, same pattern as the teaching notebook.
x = token_vectors
sublayer_output = torch.randn_like(x)

### START CODE HERE ###
# after_sublayer should be LayerNorm(x + sublayer_output)
after_sublayer = layer_norm(x + sublayer_output)
### END CODE HERE ###

print("x shape              :", x.shape)
print("sublayer output shape:", sublayer_output.shape)
print("after residual+norm  :", after_sublayer.shape)
print("mean after residual+norm:", after_sublayer.mean(dim=-1))


x shape              : torch.Size([1, 3, 4])
sublayer output shape: torch.Size([1, 3, 4])
after residual+norm  : torch.Size([1, 3, 4])
mean after residual+norm: tensor([[0., 0., 0.]], grad_fn=<MeanBackward1>)


<div dir="rtl">

### پاسخ شما — تمرین ۳

1. LayerNorm روی کدام بعد اعمال شد: batch، طول دنباله، یا ویژگی‌های همان توکن؟
2. چرا بعد از LayerNorm، توکنی که مقادیر خیلی بزرگ داشت و توکنی که مقادیر خیلی کوچک داشت، هر دو mean نزدیک صفر دارند؟

**پاسخ:**

- ۱) LayerNorm روی ویژگی های توکن اعمال میشوند .
- ۲)چون LayerNorm مقدار میانگین هر token را از تک‌تک featureهای همان token کم می‌کند.
و بعد بر انحراف معیار تقسیم می‌کند
</div>


<div dir="rtl">

# تمرین ۴ — کلاس‌های Encoder و Decoder در PyTorch

## هدف این تمرین چیست؟

در کلاس، `TinyTransformerEncoder` را با `nn.TransformerEncoderLayer` و `nn.TransformerEncoder` ساختید. Decoder را بیشتر مفهومی دیدید (<span dir="ltr">causal mask</span> + تولید توکن بعدی).

اینجا می‌خواهیم خود کلاس‌های پایتورچ را **چک کنیم** و یک مسیر کوچک <span dir="ltr">encoder–decoder</span> را با شکل تنسورها دنبال کنیم.

سناریو فروشگاه:

<pre dir="ltr">پیام مشتری  → Encoder  → memory
پاسخ پشتیبان → Decoder  → representation هر توکن پاسخ</pre>

Encoder می‌تواند به همه توکن‌های پیام مشتری نگاه کند.  
Decoder برای تولید پاسخ نباید آینده را ببیند؛ بنابراین **causal mask** لازم است.

## کاری که باید انجام دهید

**الف)** ماژول‌های داخلی `TransformerEncoderLayer` و `TransformerDecoderLayer` را چاپ کنید و تفاوت را ببینید.  
**ب)** کلاس انکودر کلاس را کامل کنید و یک batch کوچک از آن عبور دهید.  
**ج)** کلاس دیکودر را کامل کنید تا `memory` انکودر را بگیرد.  
**د)** causal mask را بسازید و چاپ کنید.

</div>


In [16]:
# Part A: inspect official PyTorch classes (no training).
d_model = 32
num_heads = 4

encoder_layer = nn.TransformerEncoderLayer(
    d_model=d_model,
    nhead=num_heads,
    dim_feedforward=4 * d_model,
    batch_first=True,
    norm_first=True,
)
decoder_layer = nn.TransformerDecoderLayer(
    d_model=d_model,
    nhead=num_heads,
    dim_feedforward=4 * d_model,
    batch_first=True,
    norm_first=True,
)

print("TransformerEncoderLayer children:")
for name, module in encoder_layer.named_children():
    print(f"  - {name:16s} {module.__class__.__name__}")

print("\nTransformerDecoderLayer children:")
for name, module in decoder_layer.named_children():
    print(f"  - {name:16s} {module.__class__.__name__}")

print("\nHint: decoder should have an extra attention module for encoder-decoder (cross) attention.")


TransformerEncoderLayer children:
  - self_attn        MultiheadAttention
  - linear1          Linear
  - dropout          Dropout
  - linear2          Linear
  - norm1            LayerNorm
  - norm2            LayerNorm
  - dropout1         Dropout
  - dropout2         Dropout

TransformerDecoderLayer children:
  - self_attn        MultiheadAttention
  - multihead_attn   MultiheadAttention
  - linear1          Linear
  - dropout          Dropout
  - linear2          Linear
  - norm1            LayerNorm
  - norm2            LayerNorm
  - norm3            LayerNorm
  - dropout1         Dropout
  - dropout2         Dropout
  - dropout3         Dropout

Hint: decoder should have an extra attention module for encoder-decoder (cross) attention.


In [20]:
# Part B: the encoder from the teaching notebook, with a few blanks.

class TinyTransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model=32, num_heads=4, num_layers=2, max_length=64):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_length, d_model)
        ### START CODE HERE ###
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,          # use the __init__ argument d_model
            nhead=num_heads,            # use num_heads
            dim_feedforward=4 * d_model,  # in class this was 4 * d_model
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer,num_layers=num_layers)        # nn.TransformerEncoder(layer, num_layers=num_layers)
        ### END CODE HERE ###

    def forward(self, token_ids):
        batch, length = token_ids.shape
        positions = torch.arange(length, device=token_ids.device).unsqueeze(0)
        ### START CODE HERE ###
        x = self.token_embedding(token_ids) + self.position_embedding(positions)  # token embedding + positional embedding
        encoded = self.encoder(x)
        ### END CODE HERE ###
        return encoded


VOCAB_SIZE = 50
encoder_model = TinyTransformerEncoder(vocab_size=VOCAB_SIZE)

# Fake customer-message token IDs: batch=2, length=6
customer_ids = torch.randint(0, VOCAB_SIZE, (2, 6))
memory = encoder_model(customer_ids)

print("customer token IDs :", customer_ids.shape)
# print(f"customer token IDs : {customer_ids}")
print("encoder memory     :", memory.shape)
# Expected: memory shape = [2, 6, 32]
# یعنی Encoder با موفقیت ورودی را گرفته و برای هر token یک بردار ویژگی ۳۲بعدی ساخته است.
# خروجی Encoder یک representation عددیِ contextualized برای هر token است.

customer token IDs : torch.Size([2, 6])
encoder memory     : torch.Size([2, 6, 32])


In [28]:
# Part C+D: decoder reads the support reply, and can also look at encoder memory.

class TinyTransformerDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=32, num_heads=4, num_layers=2, max_length=64):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_length, d_model)
        ### START CODE HERE ###
        layer = nn.TransformerDecoderLayer(
            d_model=d_model,          # same pattern as the encoder above
            nhead=num_heads,
            dim_feedforward=4 * d_model,
            batch_first=True,
            norm_first=True,
        )
        self.decoder = nn.TransformerDecoder(layer, num_layers=num_layers)        # nn.TransformerDecoder(layer, num_layers=num_layers)
        ### END CODE HERE ###

    def forward(self, reply_ids, memory):
        batch, length = reply_ids.shape
        print(f"reply_ids.shape: {reply_ids.shape}")
        print(f"memory.shape: {memory.shape}")

        positions = torch.arange(length, device=reply_ids.device).unsqueeze(0)
        print(f"positions shape: {positions.shape}")
        print(f"positions: {positions}")

        tgt = self.token_embedding(reply_ids) + self.position_embedding(positions)
        print(f"tgt: {tgt.shape}")
        # print(f"tgt: {tgt}")

        ### START CODE HERE ###
        # True means attention is blocked, same as the teaching notebook:
        # torch.triu(torch.ones(length, length, dtype=torch.bool), diagonal=1)
        causal_mask = torch.triu(torch.ones(length, length, device=reply_ids.device,dtype=torch.bool), diagonal=1)
        print(f"causal_mask: {causal_mask.shape}")
        print(f"causal_mask:\n{causal_mask}")

        decoded = self.decoder(tgt, memory,tgt_mask = causal_mask)  # self.decoder(tgt, memory, tgt_mask=causal_mask)
        print(f"decoded: {decoded.shape}")
        # print(f"decoded: {decoded}")
        ### END CODE HERE ###

        return decoded, causal_mask


decoder_model = TinyTransformerDecoder(vocab_size=VOCAB_SIZE)

# Fake support-reply token IDs: batch=2, length=5
reply_ids = torch.randint(0, VOCAB_SIZE, (2, 5))
decoded, causal_mask = decoder_model(reply_ids, memory)

print("reply token IDs :", reply_ids.shape)
print("decoder output  :", decoded.shape)
print("causal mask:\n", causal_mask)
# Expected:
#   decoder output shape = [2, 5, 32]
#   mask shape           = [5, 5]
#   upper triangle is True (future tokens blocked)


# این کد دارد یک Decoder ترنسفورمر می‌سازد تا ببینیم چگونه می‌تواند:
# توکن‌های پاسخ پشتیبانی (reply_ids) را دریافت کند.
# ترتیب توکن‌های پاسخ را با positional embedding بفهمد.
# هنگام پردازش پاسخ، به tokenهای آینده نگاه نکند.
# علاوه بر توکن‌های خودش، به خروجی Encoder یعنی memory هم دسترسی داشته باشد.

reply_ids.shape: torch.Size([2, 5])
memory.shape: torch.Size([2, 6, 32])
positions shape: torch.Size([1, 5])
positions: tensor([[0, 1, 2, 3, 4]])
tgt: torch.Size([2, 5, 32])
causal_mask: torch.Size([5, 5])
causal_mask:
tensor([[False,  True,  True,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False,  True,  True],
        [False, False, False, False,  True],
        [False, False, False, False, False]])
decoded: torch.Size([2, 5, 32])
reply token IDs : torch.Size([2, 5])
decoder output  : torch.Size([2, 5, 32])
causal mask:
 tensor([[False,  True,  True,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False,  True,  True],
        [False, False, False, False,  True],
        [False, False, False, False, False]])


<div dir="rtl">

### پاسخ شما — تمرین ۴

1. کدام ماژول فقط در Decoder دیده شد و در Encoder نبود؟ نقش آن چیست؟
2. چرا شکل خروجی Encoder برابر `[2, 6, 32]` ماند و طول دنباله عوض نشد؟
3. در causal mask، چرا خانه `(0, 4)` باید `True` باشد و خانه `(4, 0)` باید `False` باشد؟

**پاسخ:**

- ۱)cross Attention که برای مقایسه و ترکیب کوئری و کلید و مقدار به کار میرود.
masked self attention هم در انکودر صرفا self attention است.

- ۲)چون Transformer Encoder قرار نیست تعداد tokenها را تغییر بدهد؛ کارش این است که نمایش هر token را غنی‌تر کند.


- ۳)چون causal mask برای جلوگیری از دیدن آینده ساخته شده است.

در ماتریس mask:

سطر = token فعلی که می‌خواهد attention بگیرد
ستون = tokenی که آن token می‌خواهد به آن نگاه کند

</div>


<div dir="rtl">

# تمرین ۵ — از متن تمیز، مثال next-token بسازید

## هدف این تمرین چیست؟

Pretraining خودنظارتی یعنی **لیبل را از خود متن** می‌سازیم. برای یک مدل decoder، ورودی و هدف فقط یک موقعیت جابه‌جا شده‌اند:

<pre dir="ltr">input:  please refund my order
target: refund my order id</pre>

هیچ انسان دیگری لازم نیست این جفت را لیبل بزند. همین الگوی کلاس است.

## چرا این کار کاربردی است؟

اگر بعداً بخواهید یک مدل کوچک روی پیام‌های پشتیبانی pretrain کنید، اول باید از متن خام همین پنجره‌ها را بسازید.

## کاری که باید انجام دهید

1. چند پیام تمیز را با فاصله به توکن تبدیل کنید (همان tokenizer ساده کلاس).
2. واژه‌نامه کوچک بسازید.
3. پنجره‌های به طول `sequence_length + 1` بسازید.
4. `x_train` را همه توکن‌ها به‌جز آخری، و `y_train` را همه توکن‌ها به‌جز اولی بگیرید.
5. یک مثال را از ID به کلمه برگردانید و چاپ کنید.

</div>


In [44]:
pretrain_texts = [
    "please refund my order id",
    "the headphones broke after two days",
    "the box arrived on time",
    "we will replace the headphones",
]


def simple_tokenize(text: str):
    return text.split()


all_tokens = [token for doc in pretrain_texts for token in simple_tokenize(doc)]
vocab = ["<unk>"] + sorted(set(all_tokens))
stoi = {token: index for index, token in enumerate(vocab)}
itos = {index: token for token, index in stoi.items()}
token_ids = [stoi[token] for token in all_tokens]

print("vocabulary:", vocab)
print("stream of ids:", token_ids)
print("stream of tokens:", all_tokens)

sequence_length = 5
windows = []
### START CODE HERE ###
# Build windows of length sequence_length + 1, then split into input/target.
# Hint from class:
#   window = token_ids[start : start + sequence_length + 1]
#   x_train = windows[:, :-1]
#   y_train = windows[:, 1:]
for start in range(0, len(token_ids) - sequence_length, 1):
    window = token_ids[start : start + sequence_length + 1]
    windows.append(window)

windows = torch.tensor(windows, dtype=torch.long)
x_train = windows[ : , : -1]
y_train = windows[ : , 1 :]
### END CODE HERE ###

print("\ninput batch shape :", x_train.shape)
print("target batch shape:", y_train.shape)

print("=" * 60)
print("example input :", [itos[i.item()] for i in x_train[0]])
print("example target:", [itos[i.item()] for i in y_train[0]])
print("=" * 60)
print("example input :", [itos[i.item()] for i in x_train[1]])
print("example target:", [itos[i.item()] for i in y_train[1]])


vocabulary: ['<unk>', 'after', 'arrived', 'box', 'broke', 'days', 'headphones', 'id', 'my', 'on', 'order', 'please', 'refund', 'replace', 'the', 'time', 'two', 'we', 'will']
stream of ids: [11, 12, 8, 10, 7, 14, 6, 4, 1, 16, 5, 14, 3, 2, 9, 15, 17, 18, 13, 14, 6]
stream of tokens: ['please', 'refund', 'my', 'order', 'id', 'the', 'headphones', 'broke', 'after', 'two', 'days', 'the', 'box', 'arrived', 'on', 'time', 'we', 'will', 'replace', 'the', 'headphones']

input batch shape : torch.Size([16, 5])
target batch shape: torch.Size([16, 5])
example input : ['please', 'refund', 'my', 'order', 'id']
example target: ['refund', 'my', 'order', 'id', 'the']
example input : ['refund', 'my', 'order', 'id', 'the']
example target: ['my', 'order', 'id', 'the', 'headphones']


<div dir="rtl">

### پاسخ شما — تمرین ۵

1. چرا این کار self-supervised است، نه supervised با لیبل انسانی؟
2. اگر `example input` برابر `['please', 'refund', 'my', 'order', 'id']` باشد، توکن اول `example target` باید چه باشد؟

**پاسخ:**

- ۱)چون در اینجا انسان برای هر نمونه یک label تعیین نکرده است؛ خودِ متن، label را از داخل خودش می‌سازد.
- ۲)
>example input : ['please', 'refund', 'my', 'order', 'id']
>
>example target: ['refund', 'my', 'order', 'id', 'the']

</div>


<div dir="rtl">

# چک‌لیست قبل از ارسال

اگر این‌ها را دارید، تمرین کامل است:

- [ ] تمرین ۱: نظرات نرمال شده‌اند و تعداد یکتا کمتر از تعداد خام است.
- [ ] تمرین ۲: همان سه جمله با <span dir="ltr">BERT</span>، <span dir="ltr">GPT-2</span> و <span dir="ltr">XLM-RoBERTa</span> توکنایز شده و <span dir="ltr">ID</span>ها چاپ شده‌اند.
- [ ] تمرین ۳: بعد از LayerNorm، mean هر توکن نزدیک ۰ و std نزدیک ۱ است.
- [ ] تمرین ۴: `encoder memory` شکل `[2, 6, 32]` و `decoder output` شکل `[2, 5, 32]` دارد.
- [ ] تمرین ۴: causal mask مثل کلاس، مثلث بالایی `True` است.
- [ ] تمرین ۵: `y_train` همان `x_train` است که یک توکن به چپ شیفت شده.
- [ ] پنج سلول **پاسخ شما** با جواب کوتاه پر شده‌اند.

موفق باشید.

</div>
